In [0]:

from pyspark.sql.functions import regexp_replace, col
df=spark.sql("SELECT * FROM adventure_works_customers")
df = df.withColumn(
    "AnnualIncome",
    regexp_replace(col("AnnualIncome"), "[$,]", "").cast("int")
)
display(df)




In [0]:
df.filter(df.AnnualIncome > 100000).display()


In [0]:
df.filter((df.AnnualIncome > 100000)&(df.FirstName=="JOE")).display()# and condition
df.filter((df.AnnualIncome > 100000)|(df.FirstName=="JOE")).display()# or condition
df.filter(~((df.AnnualIncome > 100000)|(df.FirstName=="JOE"))).display() # not condition
df.filter(df.AnnualIncome.isNotNull()).display()# is null condition
df.filter(df.AnnualIncome.isNull()).display()# is null condition




In [0]:
df.filter(df.FirstName.startswith("J")).display()#startswith
df.filter(df.FirstName.endswith("j")).display()#endswith
df.filter(df.FirstName.contains("j")).display()#contains
df.filter(df.FirstName.like("%j%")).display()#like
df.filter(df.FirstName.rlike(".*j.*")).display()#rlike 

In [0]:
cnt=df.count()
df=df.distinct()#it will remove the duplicate rows
cnt1=df.count()
#df.display()

df2=df.dropDuplicates(['frstname','lastname'])#it will remove the duplicate rows based on the columns
cnt2=df2.count()
#df2.display()
print(cnt,cnt1,cnt2)

df.groupBy("Gender").count().display()

In [0]:
df.sort(df.AnnualIncome.desc(),df.LastName.desc()).display()
df.sort(df.AnnualIncome.asc()).display()
df.orderBy(df.AnnualIncome.desc()).display()
df.orderBy(df.AnnualIncome.asc()).display()
df.select(df.FirstName,df.LastName,df.AnnualIncome).display()
#df.select(df.FirstName.alias("fn"),df.LastName.alias("ln"),df.AnnualIncome.alias("ai")).display()
df.selectExpr("FirstName as fn","LastName as ln","AnnualIncome as ai").display()
#df.selectExpr("*","FirstName as fn","LastName as ln","

In [0]:
df.groupBy("Gender").agg({"AnnualIncome":"sum"}).display()#sum 
df.groupBy("Gender").agg({"AnnualIncome":"avg"}).display()#avg
df.groupBy("Gender").agg({"AnnualIncome":"Min","TotalChildren":"sum"}).display()#avg    
df.groupBy("Gender").agg({"AnnualIncome":"max"}).display()#max
df.groupBy("Gender").agg({"AnnualIncome":"count"}).display()#count  



In [0]:
dfcustomer=df.select(df.FirstName,df.LastName,df.AnnualIncome,df.CustomerKey)
dfotherinfo=df.select(df.CustomerKey,df.Gender,df.Occupation)
dfinner=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"inner").display()# it gives only common rows
dfleft=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"left").display()# it gives all rows from left table and null from right table

dfright=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"right").display()# it gives all rows from right table and null from left table
dfouter=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"outer").display()# it gives all rows from both table and null from other table
dfcross=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"cross").display()# it gives all rows from both table
dfcustomer=df.select(df.FirstName,df.LastName,df.AnnualIncome,df.CustomerKey)# it gives all rows from both table
dfotherinfo=df.select(df.CustomerKey,df.Gender,df.Occupation) # it gives all rows from both table
dfinner=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"inner").display()# it gives only common rows
dfleft=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"left").display()# it gives all rows from left table and null from right
dfright=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"right").display()# it gives all rows from right table and null from left
dfouter=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"outer").display()# it gives all rows from both table and null from other table
dfcross=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"cross").display( )# it gives all rows from both table
dfanti=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"anti").display()# it gives all rows from left table and null from right
dfsemileft=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"semi").display()# it gives all rows from left table and null from right
leftanti=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"leftanti").display()# it gives all rows from left table and null from right
#dfsemiright=dfcustomer.join(dfotherinfo,dfcustomer.CustomerKey==dfotherinfo.CustomerKey,"rightsemi").display()# it gives all rows from left table and null from right
dfcross=dfcustomer.crossJoin(dfotherinfo).display()


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank

window_spec = Window.partitionBy("Gender").orderBy(df.AnnualIncome.desc())

df_with_window = df.withColumn("row_num", row_number().over(window_spec)) \
                   .withColumn("rank", rank().over(window_spec)) \
                   .withColumn("dense_rank", dense_rank().over(window_spec))

display(df_with_window)